In [0]:

# Phase 3: Silver Layer - Read Bronze Data

bronze_table = "workspace.customer_data_pipeline.bronze_customers"

df_bronze = spark.table(bronze_table)

display(df_bronze)


In [0]:
Step 3: Check data quality


In [0]:

from pyspark.sql import functions as F

# Count records and missing values in Bronze
df_bronze.select(
    F.count("*").alias("total_records"),
    F.sum(F.col("email").isNull().cast("int")).alias("missing_emails"),
    F.sum(F.col("city").isNull().cast("int")).alias("missing_cities"),
    (F.count("*") - F.countDistinct("customer_id")).alias("duplicate_customer_ids")
).show()


In [0]:

# Phase 3 - Step 4: Remove duplicate customers

df_silver = df_bronze.dropDuplicates(["customer_id"])

print("Bronze record count:", df_bronze.count())
print("Silver record count:", df_silver.count())

display(df_silver.orderBy("customer_id"))


In [0]:

# Phase 3 - Step 5: Handle missing values

df_silver = df_silver.fillna({
    "email": "Unknown",
    "city": "Unknown"
})

display(df_silver.orderBy("customer_id"))


In [0]:

# Phase 3 - Step 6: Validate Silver Data

total_records = df_silver.count()

duplicate_ids = (
    total_records
    - df_silver.select("customer_id").distinct().count()
)

missing_emails = df_silver.filter(
    F.col("email").isNull()
).count()

missing_cities = df_silver.filter(
    F.col("city").isNull()
).count()

print("Silver records:", total_records)
print("Duplicate customer IDs:", duplicate_ids)
print("Missing emails:", missing_emails)
print("Missing cities:", missing_cities)

assert total_records == 4, "Incorrect Silver record count"
assert duplicate_ids == 0, "Duplicate customer IDs found"
assert missing_emails == 0, "Missing emails found"
assert missing_cities == 0, "Missing cities found"

print("SILVER DATA QUALITY VALIDATION PASSED")


In [0]:

# Phase 3 - Step 7: Write Silver Delta Table

silver_table = "workspace.customer_data_pipeline.silver_customers"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print("Silver Delta table created successfully!")
print("Table:", silver_table)


In [0]:

# Phase 3 - Step 8: Verify Saved Silver Delta Table

df_verify = spark.table(
    "workspace.customer_data_pipeline.silver_customers"
)

saved_count = df_verify.count()

print("Saved Silver records:", saved_count)

assert saved_count == 4, "Silver table validation failed!"

print("SILVER DELTA TABLE VALIDATION PASSED")

display(df_verify.orderBy("customer_id"))
